In [1]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window
from pyproj import CRS, Transformer
import plotly.graph_objects as go
import plotly.io as pio
import spiceypy as spice
pio.renderers.default = 'browser'

def plot_vector3d(fig, start, end, color="red", width =1, name="Vector"):
    """Aggiunge un vettore 3D (linea + punta a cono) a una figura esistente."""
    x0, y0, z0 = start
    x1, y1, z1 = end
    
    # 1. Linea del corpo del vettore
    fig.add_trace(go.Scatter3d(
        x=[x0, x1], y=[y0, y1], z=[z0, z1],
        mode='lines', line=dict(color=color, width=width), name=name
    ))
    
    # 2. Punta del vettore (Cono) posizionata esattamente sulla fine (end)
    fig.add_trace(go.Cone(
        x=[x1], y=[y1], z=[z1],                 # Posizione della punta
        u=[x1-x0], v=[y1-y0], w=[z1-z0],       # Direzione del vettore
        colorscale=[[0, color], [1, color]],    # Colore solido
        showscale=False, sizemode='absolute', sizeref=width/10
    ))

def plot_point3D(fig, point, color="red", size =1, name="Vector"):
    """Aggiunge un vettore 3D (linea + punta a cono) a una figura esistente."""
    x, y, z = point
    
    # 1. Linea del corpo del vettore
    fig.add_trace(go.Scatter3d(
        x=[x], y=[y], z=[z],
        mode='markers', marker=dict(size=12, color = color), name=name,
    ))
    

def plot_sphere3d(fig, center=(0,0,0), radius=1.0, color="lightblue", name="Sphere"):
    """Aggiunge una sfera 3D a una figura Plotly esistente."""
    xc, yc, zc = center
    
    # 1. Genera la griglia di coordinate sferiche (angoli)
    phi = np.linspace(0, np.pi, 30)
    theta = np.linspace(0, 2 * np.pi, 30)
    phi, theta = np.meshgrid(phi, theta)
    
    # 2. Trasforma in coordinate cartesiane e applica raggio e centro
    x = xc + radius * np.sin(phi) * np.cos(theta)
    y = yc + radius * np.sin(phi) * np.sin(theta)
    z = zc + radius * np.cos(phi)
    
    # 3. Aggiunge la superficie alla figura con un colore solido personalizzato
    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        colorscale=[[0, color], [1, color]],
        showscale=False,
        opacity=0.4,
        name=name
    ))

def lla_to_pa_xyz(lat_deg, lon_deg, alt_m, R, utc_time="2026-06-22T00:00:00"):
    xyz_me = lla_to_moon_me(lat_deg, lon_deg, alt_m, R)

    et = spice.str2et(utc_time)
    R_me_to_pa = spice.pxform("MOON_ME", "MOON_PA", et)

    xyz_pa = R_me_to_pa @ xyz_me

    return xyz_pa

def lla_to_moon_me(lat_deg, lon_deg, alt_m, R):
    """
    Convert lunar latitude/longitude/altitude to Moon-centered Cartesian XYZ.

    Input:
        lat_deg: latitude, degrees
        lon_deg: longitude, degrees
        alt_m: altitude above reference lunar sphere, meters

    Output:
        [X, Y, Z] in meters, origin at Moon center
    """
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)

    r = R + alt_m

    X = r * np.cos(lat) * np.cos(lon)
    Y = r * np.cos(lat) * np.sin(lon)
    Z = r * np.sin(lat)

    return np.array([X, Y, Z], dtype=float)

def inspect_dem(dem_path):
    with rasterio.open(dem_path) as src:
        print("CRS:")
        print(src.crs)
        print()

        print("Size:")
        print(f"width  = {src.width}")
        print(f"height = {src.height}")
        print()

        print("Resolution:")
        print(src.res)
        print()

        print("Bounds:")
        print(src.bounds)
        print()

        print("Transform:")
        print(src.transform)
        print()

        print("NoData:")
        print(src.nodata)

R_LOLA_M = 1_737_400.0

def dem_to_moon_pa(
    dem_path,
    downsample_factor=100,
    r_moon=R_LOLA_M,
    dem_crs_fallback=None,
    resampling=Resampling.nearest,
    utc_time="2026-06-22T00:00:00",
    compute_pa=True,
):
    """
    Convert a LOLA polar stereographic DEM to Moon-centered Cartesian XYZ.

    Output:
        X_me, Y_me, Z_me:
            DEM wrapped onto Moon-centered Mean Earth/body-fixed-like frame,
            in meters.

        X_pa, Y_pa, Z_pa:
            Same points rotated into MOON_PA, if compute_pa=True.
            Otherwise these are returned as None.

        dem:
            Downsampled DEM elevation values [m].

        x_ps, y_ps:
            DEM polar stereographic map coordinates [m].

        lat_deg, lon_deg:
            Lunar latitude/longitude recovered from the DEM projection.
    """

    # ------------------------------------------------------------
    # 1. Read DEM and its real georeferencing metadata
    # ------------------------------------------------------------
    with rasterio.open(dem_path) as src:
        src_crs = src.crs

        if src_crs is None:
            if dem_crs_fallback is None:
                raise ValueError(
                    "DEM has no CRS. Provide dem_crs_fallback explicitly."
                )
            dem_crs = dem_crs_fallback
        else:
            dem_crs = src_crs

        out_height = src.height // downsample_factor
        out_width = src.width // downsample_factor

        if out_height < 2 or out_width < 2:
            raise ValueError("downsample_factor is too large for this DEM.")

        dem = src.read(
            1,
            out_shape=(out_height, out_width),
            resampling=resampling,
        ).astype(float)

        # Transform valid for the downsampled grid
        transform = src.transform * src.transform.scale(
            src.width / out_width,
            src.height / out_height,
        )

        nodata = src.nodata

    if nodata is not None:
        dem[dem == nodata] = np.nan

    # ------------------------------------------------------------
    # 2. Raster row/col -> DEM projected x/y
    # ------------------------------------------------------------
    rows, cols = np.indices(dem.shape)

    x_ps = (
        transform.c
        + (cols + 0.5) * transform.a
        + (rows + 0.5) * transform.b
    )

    y_ps = (
        transform.f
        + (cols + 0.5) * transform.d
        + (rows + 0.5) * transform.e
    )

    # ------------------------------------------------------------
    # 3. DEM projected x/y -> lunar lon/lat
    # ------------------------------------------------------------
    lunar_latlon = CRS.from_proj4(
        f"+proj=longlat +R={r_moon} +no_defs +type=crs"
    )

    transformer = Transformer.from_crs(
        dem_crs,
        lunar_latlon,
        always_xy=True,
    )

    lon_deg, lat_deg = transformer.transform(x_ps, y_ps)

    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)

    # ------------------------------------------------------------
    # 4. lon/lat/elevation -> Moon-centered Cartesian XYZ
    # ------------------------------------------------------------
    radius = r_moon + dem

    X_me = radius * np.cos(lat) * np.cos(lon)
    Y_me = radius * np.cos(lat) * np.sin(lon)
    Z_me = radius * np.sin(lat)

    # Preserve invalid pixels as NaN
    invalid = np.isnan(dem)
    X_me[invalid] = np.nan
    Y_me[invalid] = np.nan
    Z_me[invalid] = np.nan

    X_pa = Y_pa = Z_pa = None

    if compute_pa:
        xyz_me_m = np.column_stack([
            X_me.ravel(),
            Y_me.ravel(),
            Z_me.ravel(),
        ])

        valid = ~np.any(np.isnan(xyz_me_m), axis=1)

        xyz_pa_m = np.full_like(xyz_me_m, np.nan)

        et = spice.str2et(utc_time)
        R_me_to_pa = spice.pxform("MOON_ME", "MOON_PA", et)

        xyz_pa_m[valid] = xyz_me_m[valid] @ R_me_to_pa.T

        X_pa = xyz_pa_m[:, 0].reshape(dem.shape)
        Y_pa = xyz_pa_m[:, 1].reshape(dem.shape)
        Z_pa = xyz_pa_m[:, 2].reshape(dem.shape)

    return X_me, Y_me, Z_me, X_pa, Y_pa, Z_pa, dem, x_ps, y_ps, lat_deg, lon_deg

spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/naif0012.tls")
spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/moon_de440_220930.tf")
spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/moon_pa_de440_200625.bpc")
spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/pck00011.tpc")

In [2]:
r_moon = 1_737_400.0

lat_deg = 45
lon_deg = 45
alt_m = 0
radius = r_moon
pos = lla_to_pa_xyz(lat_deg, lon_deg, alt_m, radius)

start = np.array([0, 0, 0])
end = pos

fig = go.Figure()

plot_vector3d(fig, start, end, color = 'purple', width = 3 , name="Freccia")
plot_vector3d(fig, start, lla_to_pa_xyz(0, 0, 0, radius), color = 'red', width = 3 , name="X")
plot_vector3d(fig, start, lla_to_pa_xyz(0, 90, 0, radius), color = 'green', width = 3 , name="Y")
plot_vector3d(fig, start, lla_to_pa_xyz(90, 0, 0, radius), color = 'blue', width = 3 , name="Z")

plot_sphere3d(fig, start, radius = radius, color = "grey", name = "Sfera")

fig.show()


In [3]:
dem_path = r"/Users/pierazhi/Desktop/Multipath/Multipath/dem2mesh/dems/LDEM_45S_100M.JP2"
# inspect_dem(dem_path)

X_me, Y_me, Z_me, X_pa, Y_pa, Z_pa, dem, x_ps, y_ps, lat_deg, lon_deg = dem_to_moon_pa(
    dem_path,
    downsample_factor=25,
)

In [4]:
fig = go.Figure()

plot_vector3d(fig, start, lla_to_pa_xyz(-45, 0, 0, radius), color = 'purple', width = 3 , name="Freccia")
plot_vector3d(fig, start, lla_to_pa_xyz(0, 0, 0, radius), color = 'red', width = 3 , name="X")
plot_vector3d(fig, start, lla_to_pa_xyz(0, 90, 0, radius), color = 'green', width = 3 , name="Y")
plot_vector3d(fig, start, lla_to_pa_xyz(90, 0, 0, radius), color = 'blue', width = 3 , name="Z")
plot_point3D(fig, lla_to_pa_xyz(-89.67, 129.78, 10, radius), color = 'yellow', size = 5, name = "Shackleton Crater")
plot_point3D(fig, lla_to_pa_xyz(-88.48, 271.6533, 10, radius), color = 'blue', size = 5, name = "de Gerlache Crater")
plot_point3D(fig, lla_to_pa_xyz(-85.99, -2.93, 10, radius), color = 'green', size = 5, name = "Malapert Massif")
plot_point3D(fig, lla_to_pa_xyz(-88.18907, 79.50852, 10, radius), color = 'red', size = 5, name = "Faustini Rim A")
plot_point3D(fig, lla_to_pa_xyz(-84.50, 82.80, 10, radius), color = 'purple', size = 5, name = "Amundsen Crater")
plot_point3D(fig, lla_to_pa_xyz(-84.90, -35.50, 10, radius), color = 'magenta', size = 5, name = "Cabeus Crater")


plot_sphere3d(fig, start, radius = radius, color = "grey", name = "Sfera")
fig.add_trace(go.Surface(
        x=X_me, 
        y=Y_me, 
        z=Z_me,
        
        # Passiamo la coordinata Z o i dati di altezza grezzi per decidere il colore.
        # Se la patch è molto piccola, puoi usare Z_mesh direttamente.
        # surfacecolor=Z_mesh, 
        
        # colorscale='Viridis', 
        
        colorscale=[[0, 'grey'], [1, 'grey']],
        showscale=False,        # Mostra la barra della legenda dei colori a destra con le altezze
        opacity=1.0,           
        
        name="Terreno Lunare",

        lighting=dict(
        ambient=0.4,
        diffuse=0.8,
        fresnel=0.2,
        specular=0.1,
        roughness=0.5
        ),
        lightposition=dict(
            x=0,
            y=0,
            z=-5  # Posiziona una luce virtuale vicino al Polo Sud per creare le ombre nei crateri
        )
    
    ))

    
fig.update_layout(
    scene=dict(
        aspectmode='data',
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)'
    )
)
fig.show()